# E9: telling Whisper which names to expect

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivannamor/multilingual-asr-lab/blob/main/notebooks/e9_whisper_colab.ipynb)

Experiment E9 of [RESULTS.md](https://github.com/ivannamor/multilingual-asr-lab/blob/main/RESULTS.md)
gives large-v3-turbo a list of named entities as its initial prompt: the entities of the utterance
(an upper bound), those of another utterance (the harm of a wrong list) and, on VoxPopuli, the
commonest ones of the training pool (the realistic case).

The notebook has two stages, on purpose, so that the test sets are decoded only with settings
already fixed:

- `SPLIT = "dev"`: three runs on the VoxPopuli validation split, to choose the size of the list.
- `SPLIT = "test"`: every list on the test sets, after the selection has been merged into `main`.

1. On Google Colab: *Runtime → Change runtime type → T4 GPU*, the GPU of the runs without a list.
   Set `SPLIT` below, then *Runtime → Run all* and keep the tab open. The first run installs the
   package and restarts the session: then *Run all* again. Colab reports this as a crash "for an
   unknown reason": it is expected.
2. At the end the browser downloads `e9_runs_<split>.zip` (if it does not, download it from the
   *Files* panel): the predictions and `run.json` of every transcription, no audio and no weights.
3. In a local clone, from the repository root: `python3 -m zipfile -e e9_runs_<split>.zip .`

If the session is interrupted but the machine survives, run all cells again: a transcription
resumes where it stopped. A new machine starts from zero, because the runs are on its local disk.


## 1. Clone and install


In [ ]:
import importlib.util
import os
import subprocess
import sys
import time
from pathlib import Path

# Work inside a clone of the repository, so that every run.json records the git commit and the
# run directories end up where the experiment scripts expect them.
if Path("../pyproject.toml").exists():  # started from notebooks/ in a local clone
    os.chdir("..")
elif not Path("pyproject.toml").exists():  # on Colab
    if not Path("multilingual-asr-lab").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ivannamor/multilingual-asr-lab"],
            check=True,
        )
    os.chdir("multilingual-asr-lab")
if importlib.util.find_spec("asr_lab") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "."], check=True)
    # pip may have upgraded packages that the kernel has already imported (numpy), and mixing two
    # versions in one process fails later: restart the session, then run all the cells again
    # (this one will skip the installation)
    # Colab reports this as a crash "for an unknown reason": it is expected. The pause lets the
    # message reach the notebook before the kernel dies.
    print("Installed: the session restarts. Run all the cells again.", flush=True)
    time.sleep(2)
    os.kill(os.getpid(), 9)
print(Path.cwd())

## 2. Settings


In [ ]:
SPLIT = "dev"  # "dev", then "test" once the selection is in main
SPEC_PATH = Path("experiments/E9/biasing.toml")
ENTITIES_PATH = Path("experiments/E9/results/entities.json")
SELECTION_PATH = Path("experiments/E9/results/selection.json")
RUNS_DIR = Path("experiments/runs")

## 3. Data

The audio of the chosen split: on dev the VoxPopuli validation split (about 0.9 GB), on test the three
FLEURS test sets (about 1.5 GB) and the VoxPopuli test split (about 0.9 GB).


In [ ]:
import json

from asr_lab.biasing.experiment import whisper_plan
from asr_lab.data.ingest import ingest_fleurs
from asr_lab.data.voxpopuli import ingest_voxpopuli, load_data_spec
from asr_lab.lm.corpus import load_spec

spec = load_spec(SPEC_PATH)
entities_report = json.loads(ENTITIES_PATH.read_text(encoding="utf-8"))
selection = json.loads(SELECTION_PATH.read_text(encoding="utf-8")) if SPLIT == "test" else None
voxpopuli = {"dev": "experiments/E4/voxpopuli_dev.toml", "test": "experiments/E7/test.toml"}
plan = whisper_plan(spec, entities_report, SPLIT, selection)
for test_set in {t["name"]: t for t, _, _ in plan}.values():
    manifest = Path(test_set[SPLIT])
    if not manifest.exists():
        if test_set["name"].startswith("fleurs"):
            ingest_fleurs(test_set["lang"], SPLIT, manifest.parent)
        else:
            ingest_voxpopuli(load_data_spec(Path(voxpopuli[SPLIT])))
    print(manifest)
print(f"{len(plan)} runs")

### GPU setup for Whisper

faster-whisper needs the CUDA 12 libraries that CTranslate2 is built for, loaded explicitly.


In [ ]:
import ctypes

import ctranslate2
import numpy as np

if ctranslate2.get_cuda_device_count() == 0:
    raise RuntimeError("No GPU: choose Runtime -> Change runtime type -> T4 GPU")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"],
    check=True,
)

In [ ]:
import nvidia.cublas
import nvidia.cudnn
from faster_whisper import WhisperModel

pending = [
    path
    for package in (nvidia.cublas, nvidia.cudnn)
    for path in sorted(Path(next(iter(package.__path__))).glob("lib/*.so*"))
]
for _ in range(5):  # the libraries depend on each other: retry the ones that failed
    failed = []
    for path in pending:
        try:
            ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL)
        except OSError:
            failed.append(path)
    pending = failed
    if not pending:
        break

check = WhisperModel("tiny", device="cuda", compute_type="float16")
list(check.transcribe(np.zeros(16_000, dtype=np.float32), language="it")[0])
print("GPU ready")

## 4. Transcription


In [ ]:
from asr_lab.biasing.experiment import run_whisper

run_dirs = run_whisper(
    SPEC_PATH, ENTITIES_PATH, SPLIT, RUNS_DIR, SELECTION_PATH if SPLIT == "test" else None, "cuda"
)

## 5. Download the results


In [ ]:
import zipfile

archive = Path(f"e9_runs_{SPLIT}.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for run_dir in run_dirs:
        for file in ("predictions.jsonl", "run.json"):
            zipped.write(run_dir / file, (run_dir / file).as_posix())
        run = json.loads((run_dir / "run.json").read_text())
        print(f"{run_dir.name}: RTF {run['rtf'] or float('nan'):.3f}")
print(f"{archive.resolve()}: {archive.stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import files

    files.download(str(archive))
except ImportError:  # not on Colab
    pass